# 04 — Holiday effects

Compare Prophet without vs with holiday regressors. Forecasts around holiday dates and error reduction show why holiday effects matter for surgery planning.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.evaluation import train_val_split, mae, mape
data_path = root / "data" / "surgeries.csv"
if not data_path.exists():
    from src.data_generator import generate_surgery_counts
    generate_surgery_counts(output_path=str(data_path))
df = pd.read_csv(data_path, parse_dates=["date"])
series = df[(df["specialty_name"] == "General Surgery") & (df["hospital_id"] == "H1")][["date", "surgery_count"]].sort_values("date").reset_index(drop=True)
train, val = train_val_split(series, date_col="date", val_days_or_ratio=0.2)
y_val = val["surgery_count"].values
n_val = len(y_val)

## Prophet without holidays

In [ ]:
from prophet import Prophet
train_p = train[["date", "surgery_count"]].rename(columns={"date": "ds", "surgery_count": "y"})
m_no_hol = Prophet(yearly_seasonality=True, weekly_seasonality=True)
m_no_hol.fit(train_p)
future = m_no_hol.make_future_dataframe(periods=n_val)
forecast_no = m_no_hol.predict(future)
pred_no_hol = forecast_no["yhat"].tail(n_val).values
print("Without holidays: MAE =", round(mae(y_val, pred_no_hol), 2), " MAPE =", round(mape(y_val, pred_no_hol, skip_zeros=True), 1), "%")

## Prophet with holidays

In [ ]:
thanksgiving = []
for y in range(2021, 2025):
    nov1 = pd.Timestamp(f"{y}-11-01")
    thu = nov1 + pd.Timedelta(days=(3 - nov1.dayofweek) % 7 + 21)
    thanksgiving.append(thu.strftime("%Y-%m-%d"))
holidays_df = pd.DataFrame({
    "holiday": ["ny"]*4 + ["july4"]*4 + ["xmas"]*4 + ["nye"]*4 + ["thanksgiving"]*4,
    "ds": pd.to_datetime(
        [f"{y}-01-01" for y in range(2021,2025)] + [f"{y}-07-04" for y in range(2021,2025)] +
        [f"{y}-12-25" for y in range(2021,2025)] + [f"{y}-12-31" for y in range(2021,2025)] + thanksgiving
    ),
})
m_hol = Prophet(holidays=holidays_df, yearly_seasonality=True, weekly_seasonality=True)
m_hol.fit(train_p)
forecast_hol = m_hol.predict(future)
pred_hol = forecast_hol["yhat"].tail(n_val).values
print("With holidays:   MAE =", round(mae(y_val, pred_hol), 2), " MAPE =", round(mape(y_val, pred_hol, skip_zeros=True), 1), "%")

## Forecasts around holiday dates

In [ ]:
val_dates = val["date"].values
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(val_dates, y_val, label="Actual", alpha=0.8)
ax.plot(val_dates, pred_no_hol, label="Prophet (no holidays)", alpha=0.8)
ax.plot(val_dates, pred_hol, label="Prophet + holidays", alpha=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Surgery count")
ax.set_title("Holiday effect: with vs without holiday regressors")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()